# Study 9: marginal likelihood of three linking / β models

Study 9 preregistered data, all fits already cached (nothing is refit here):

| model | linking function | speaker β | fit |
|---|---|---|---|
| **ZOIB, β fitted** | per-feature ZOIB (`linking.py`) | fitted | `results/study9-fit-zoib/posterior.npz` |
| **original, β fixed** | per-feature Beta mixture | fixed at study 6's median (≈28.9) | `results/study9-fit-fixed-beta/fitted-feature-shapes.npz` |
| **original, β fitted** | per-feature Beta mixture | fitted | `results/study9-fit/fitted-feature-shapes.npz` |

Each also has a distance-blind null (length scale pinned at 10), shown in the supplementary table.

**The ELBOs can't be compared as they come out of VBMC.** They're in different units:
- the **original** linking scores every rating as a Beta **density** on [0, 1], after moving exact 0s and 100s half a
  slider bin inward;
- the **ZOIB** linking scores exact 0s and 100s as **probabilities** (its point masses) and the other ratings as
  densities.

Densities and probabilities differ by a factor on the order of the bin width (0.01), per rating. With 6,015 ratings,
734 of them exact 0 or 100, the raw ELBOs end up thousands of nats apart for reasons that have nothing to do with fit.

**Fix: score both as discrete distributions over the 101 slider values.** A rating of k is given the probability of
its slider bin [(k − ½)/100, (k + ½)/100], clipped to [0, 1], from each component's Beta CDF. For ZOIB, the
point masses are added to the 0 and 100 bins. Both models then give a proper distribution over the same 101
outcomes, so their log-likelihoods are in the same units. The discrete log-likelihoods are plugged into the same
Laplace machinery (`inference.total_log_lik`) in place of the density ones, so this gives an **exact discretized
Laplace log Z** at a given θ, not an approximation.

**Converting the ELBO.** The ELBO is E_q[log joint(θ)] + H[q], with q the VBMC posterior. Swapping the likelihood
changes the log joint at each θ by shift(θ) = log Z_discrete(θ) − log Z_density(θ), so for the same q,
ELBO_discrete = ELBO + E_q[shift(θ)]. That's exact apart from Monte Carlo error; here E_q is estimated from
N_DRAWS posterior draws, and the standard error is reported. It's still a valid lower bound on the discrete
evidence, just with q optimized for the density likelihood, not re-optimized (which could only raise it).

1. Native numbers (not comparable across linking functions)
2. Discretized log Z and converted ELBO (comparable)
   - 2b. Fitted length scale vs distance-blind null, per model (log Bayes factor)
3. How much the shift varies across the posterior
4. Supplementary: nulls and β = 3
5. Caveats

In [1]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import json
import numpy as np
import pandas as pd
import jax.numpy as jnp
from scipy.special import betainc

import inference as inf
import linking as lk

geom = inf.load_geometry()
CONDS = list(inf.CONDITIONS)
resp_nudged, _ = inf.load_responses('../../data/study9_prereg.csv')                    # original linking
resp_raw, _ = inf.load_responses('../../data/study9_prereg.csv', clip_interior=False)  # ZOIB, and the bins
K = {c: np.round(np.asarray(resp_raw[c]) * 100).astype(int) for c in CONDS}           # slider values 0-100
Kall = np.concatenate([K[c] for c in CONDS])
N = Kall.size
print(f"{N} ratings; exact 0: {(Kall == 0).sum()}, exact 100: {(Kall == 100).sum()}, "
      f"interior: {((Kall > 0) & (Kall < 100)).sum()}")

THETA_KEYS = ('ls_samples', 'mu0_samples', 'sigma_samples', 'beta_samples')
MODELS = {   # name: (linking, posterior file)
    'ZOIB, β fitted':         ('zoib', 'results/study9-fit-zoib/posterior.npz'),
    'original, β fixed (≈29)': ('orig', 'results/study9-fit-fixed-beta/fitted-feature-shapes.npz'),
    'original, β fitted':     ('orig', 'results/study9-fit/fitted-feature-shapes.npz'),
}
NULLS = {
    'ZOIB, β fitted: null':          ('zoib', 'results/study9-fit-zoib/posterior-null.npz'),
    'original, β fixed (≈29): null': ('orig', 'results/study9-fit-fixed-beta/fitted-feature-shapes-null.npz'),
    'original, β fitted: null':      ('orig', 'results/study9-fit/fitted-feature-shapes-null.npz'),
    'original, β fixed at 3':        ('orig', 'results/study9-fit-beta3/fitted-feature-shapes.npz'),
    'original, β fixed at 3: null':  ('orig', 'results/study9-fit-beta3/fitted-feature-shapes-null.npz'),
}

def load_post(path):
    z = np.load(path, allow_pickle=True)
    return {k: np.asarray(z[k]) for k in THETA_KEYS}, json.loads(str(z['meta']))

6015 ratings; exact 0: 34, exact 100: 700, interior: 5281


### Discrete (per-slider-value) log-likelihoods

`bin_logprob_beta(k, a, b)` is log P(slider = k) under a plain Beta: its CDF mass over k's bin. The original
component is exactly that. The ZOIB component is π0 at 0, π1 at 100, plus (1 − π0 − π1) × the Beta's bin mass
(its Beta mass in the edge half-bins goes to 0 and 100 too, so each component sums to 1 over the 101 values).

In [2]:
LO = lambda k: np.clip((k - 0.5) / 100, 0, 1)
HI = lambda k: np.clip((k + 0.5) / 100, 0, 1)

def bin_mass_beta(k, a, b):
    return np.maximum(betainc(a, b, HI(k)) - betainc(a, b, LO(k)), 1e-300)

def disc_orig(k, shapes):
    """(log P(k | kl), log P(k | nkl)) for the original linking; shapes (J, 4)."""
    a_kl, b_kl, a_nkl, b_nkl = shapes.T
    return np.log(bin_mass_beta(k, a_kl, b_kl)), np.log(bin_mass_beta(k, a_nkl, b_nkl))

def disc_zoib(k, link_theta):
    p = lk.params_to_numpy(lk.unpack(jnp.asarray(link_theta)))
    out = []
    for c in ('kl', 'nkl'):
        w = 1 - p['pi0_' + c] - p['pi1_' + c]
        m = w * bin_mass_beta(k, p['a_' + c], p['b_' + c])
        m = m + np.where(k == 0, p['pi0_' + c], 0) + np.where(k == 100, p['pi1_' + c], 0)
        out.append(np.log(m))
    return out

# each component is a proper distribution over the 101 values
ks = np.arange(101)[:, None]
_sh = np.load('results/study9-fit/fitted-link-shapes.npz')['shapes']
_lt = np.load('results/study9-fit-zoib/fitted-link-zoib.npz')['link_theta']
for name, (lkl, lnkl) in [('original', disc_orig(ks, _sh)), ('ZOIB', disc_zoib(ks, _lt))]:
    print(f"{name}: max |sum_k P(k) - 1| = kl {np.abs(np.exp(lkl).sum(0) - 1).max():.1e}, "
          f"nkl {np.abs(np.exp(lnkl).sum(0) - 1).max():.1e}")

original: max |sum_k P(k) - 1| = kl 0.0e+00, nkl 0.0e+00
ZOIB: max |sum_k P(k) - 1| = kl 4.4e-16, nkl 4.4e-16


### Scoring one model at one θ

Profile the model's linking function at θ as its fit does, then compute Laplace log Z twice from the same
`cond_data`: with the native per-rating log-likelihoods (this reproduces the fit's own objective), and with
the discrete ones swapped in.

In [3]:
def score(link, theta):
    ls, mu, sig, b = theta
    if link == 'orig':
        ll_dens, shapes, cd = inf.total_log_lik_free_shapes(geom, resp_nudged, ls, mu, sig,
                                                            beta_speaker=b, return_shapes=True)
        disc = lambda c: disc_orig(K[c], shapes)
    else:
        ll_dens, link_theta, cd = inf.total_log_lik_zoib(geom, resp_raw, ls, mu, sig,
                                                         beta_speaker=b, return_link=True)
        disc = lambda c: disc_zoib(K[c], link_theta)
    cd_disc = {}
    for c in CONDS:
        lkl, lnkl = disc(c)
        cd_disc[c] = {**cd[c], 'logpdf_kl': jnp.asarray(lkl), 'logpdf_nkl': jnp.asarray(lnkl)}
    ll_disc = float(inf.total_log_lik(cd_disc, ls, mu, sig, b))
    return float(ll_dens), ll_disc

median_theta = lambda post: tuple(float(np.median(post[k])) for k in THETA_KEYS)
N_DRAWS = 40
SHIFTS = {}

def summarize(models, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for name, (link, path) in models.items():
        if not os.path.exists(path):
            print(f"  (skipping {name}: {path} not found)"); continue
        post, meta = load_post(path)
        th = median_theta(post)
        dens, disc = score(link, th)
        idx = rng.choice(len(post['ls_samples']), N_DRAWS, replace=False)
        sh = np.array([np.subtract(*score(link, tuple(float(post[k][i]) for k in THETA_KEYS))[::-1]) for i in idx])
        SHIFTS[name] = sh
        rows.append({'model': name, 'ls': th[0], 'μ₀': th[1], 'σ': th[2], 'β': th[3],
                     'VBMC status': meta['convergence_status'], 'evals': meta['func_count'],
                     'ELBO (native)': meta['elbo'], 'log Z @ median (native)': dens,
                     'log Z @ median (discrete)': disc, 'shift @ median': disc - dens,
                     'E_q[shift]': sh.mean(), 'SE': sh.std(ddof=1) / np.sqrt(len(sh)),
                     'ELBO (discrete, converted)': meta['elbo'] + sh.mean()})
    return pd.DataFrame(rows).set_index('model')

main = summarize(MODELS)

## 1. Native numbers (not comparable across linking functions)

As VBMC and the Laplace approximation report them. The two original-linking rows can be compared with each
other; the ZOIB row can't be compared with either.

In [4]:
main[['ls', 'μ₀', 'σ', 'β', 'VBMC status', 'evals', 'ELBO (native)', 'log Z @ median (native)']].round(3)

,ls,μ₀,σ,β,VBMC status,evals,ELBO (native),log Z @ median (native)
model,,,,,,,,
"ZOIB, β fitted",0.344,1.293,0.522,25.931,probable,195,-1724.912,-1719.893
"original, β fixed (≈29)",1.824,-0.231,0.286,28.948,probable,250,974.532,981.557
"original, β fitted",0.978,0.006,0.526,0.907,no,400,983.342,988.751


## 2. Discretized log Z and converted ELBO (comparable)

All three in the same units: log probability of the observed slider values. Higher is better; Δ is against
the best model. The **converted ELBO** is the model-evidence comparison (it integrates over θ); log Z at the
median is the same comparison at a single θ.

In [5]:
tab = main[['ELBO (native)', 'E_q[shift]', 'SE', 'ELBO (discrete, converted)', 'log Z @ median (discrete)']].copy()
tab['Δ ELBO vs best'] = tab['ELBO (discrete, converted)'] - tab['ELBO (discrete, converted)'].max()
tab['Δ log Z vs best'] = tab['log Z @ median (discrete)'] - tab['log Z @ median (discrete)'].max()
tab.sort_values('ELBO (discrete, converted)', ascending=False).round(1)

,ELBO (native),E_q[shift],SE,"ELBO (discrete, converted)",log Z @ median (discrete),Δ ELBO vs best,Δ log Z vs best
model,,,,,,,
"ZOIB, β fitted",-1724.9,-24310.3,0.0,-26035.2,-26030.1,0.0,0.0
"original, β fitted",983.3,-27420.1,1.1,-26436.8,-26436.8,-401.6,-406.7
"original, β fixed (≈29)",974.5,-27417.3,1.5,-26442.8,-26438.6,-407.6,-408.5


### 2b. Fitted length scale vs distance-blind null, per model

Each model has a null fit with the same linking function and the length scale pinned at `LS_FLAT` = 10, so
coherence can't depend on distance in the embedding. The null's other parameters are set up like its full
model: β is fitted in the null too where the full model fits it, and fixed at the same value where it's fixed.
The β = 3 fit is included as a fourth pair.

Δ = full − null (positive = the fitted length scale wins). **Δ ELBO is the estimated log Bayes factor**, and
Δ log Z @ median is the plug-in version, with no Occam penalty for the extra ls parameter. The native columns
are what VBMC reports. For ZOIB, native and discrete agree (point masses are scored the same way in both units).
For the original linking they don't: the native numbers score the exact 0s and 100s by density at the nudged
value, and the discrete ones score them by the probability of the slider value (section 3). Read the discrete
columns. ± is the Monte Carlo SE of the discrete ELBO difference. As a rough scale, |log BF| > 3 is usually
read as strong evidence and > 5 as very strong.

In [6]:
PAIRS = {   # full model: its null
    'ZOIB, β fitted':          'ZOIB, β fitted: null',
    'original, β fitted':      'original, β fitted: null',
    'original, β fixed (≈29)': 'original, β fixed (≈29): null',
    'original, β fixed at 3':  'original, β fixed at 3: null',
}
supp = summarize(NULLS, seed=1)
both = pd.concat([main, supp])
rows = []
for f, n in PAIRS.items():
    if f not in both.index or n not in both.index:
        print(f"  (skipping {f}: missing fit)"); continue
    F, Nl = both.loc[f], both.loc[n]
    rows.append({'model': f, 'β (full / null)': f"{F['β']:.2f} / {Nl['β']:.2f}",
                 'VBMC status (full / null)': f"{F['VBMC status']} / {Nl['VBMC status']}",
                 'ls (full)': F['ls'],
                 'Δ ELBO native': F['ELBO (native)'] - Nl['ELBO (native)'],
                 'Δ log Z @ median native': F['log Z @ median (native)'] - Nl['log Z @ median (native)'],
                 'Δ ELBO discrete (log BF)': F['ELBO (discrete, converted)'] - Nl['ELBO (discrete, converted)'],
                 '±': np.hypot(F['SE'], Nl['SE']),
                 'Δ log Z @ median discrete': F['log Z @ median (discrete)'] - Nl['log Z @ median (discrete)']})
fvn = pd.DataFrame(rows).set_index('model')
fvn['favors'] = np.where(fvn['Δ ELBO discrete (log BF)'] > 0, 'fitted ls', 'null')
fvn.round(1)

,β (full / null),VBMC status (full / null),ls (full),Δ ELBO native,Δ log Z @ median native,Δ ELBO discrete (log BF),±,Δ log Z @ median discrete,favors
model,,,,,,,,,
"ZOIB, β fitted",25.93 / 22.80,probable / probable,0.3,6.9,6.7,7.0,0.0,6.9,fitted ls
"original, β fitted",0.91 / 0.73,no / probable,1.0,13.3,15.6,-17.6,2.0,-27.3,null
"original, β fixed (≈29)",28.95 / 28.95,probable / probable,1.8,11.2,12.8,-10.5,2.3,-7.1,null
"original, β fixed at 3",3.00 / 3.00,no / probable,1.4,10.4,1.4,-1.9,2.7,4.8,null


**Summary: Bayes factor, fitted length scale vs null, per model.** log BF(fitted ls : null) =
ELBO(full) − ELBO(null), in common units (slider-value probabilities). **Positive favors the fitted length
scale**; negative favors the null (ls = 10, distance-blind). BF = exp(log BF). ± is the Monte Carlo SE; the
verdict reads "unclear" when |log BF| is under 2 SEs.

In [7]:
SPEC = {   # model: (linking, rationality)
    'ZOIB, β fitted':          ('ZOIB', 'fitted'),
    'original, β fitted':      ('original', 'fitted'),
    'original, β fixed (≈29)': ('original', 'fixed (β = 28.9, study 6)'),
    'original, β fixed at 3':  ('original', 'fixed (β = 3)'),
}
lbf = fvn['Δ ELBO discrete (log BF)']
summary = pd.DataFrame({
    'linking': [SPEC[m][0] for m in fvn.index],
    'speaker rationality (β)': [SPEC[m][1] for m in fvn.index],
    'log BF (fitted ls : null)': [f"{b:+.1f} ± {e:.1f}" for b, e in zip(lbf, fvn['±'])],
    'BF (fitted ls : null)': [f"{np.exp(b):.2g}" for b in lbf],
    'favors': np.where(np.abs(lbf) < 2 * fvn['±'], 'unclear', np.where(lbf > 0, 'fitted ls', 'null')),
}, index=pd.Index(fvn.index, name='model'))
summary

,linking,speaker rationality (β),log BF (fitted ls : null),BF (fitted ls : null),favors
model,,,,,
"ZOIB, β fitted",ZOIB,fitted,+7.0 ± 0.0,1.1e+03,fitted ls
"original, β fitted",original,fitted,-17.6 ± 2.0,2.3e-08,null
"original, β fixed (≈29)",original,"fixed (β = 28.9, study 6)",-10.5 ± 2.3,2.6e-05,null
"original, β fixed at 3",original,fixed (β = 3),-1.9 ± 2.7,0.15,unclear


## 3. How much the shift varies across the posterior

Spread of shift(θ) over the N_DRAWS posterior draws used for E_q[shift]. For ZOIB it's nearly constant (its
point masses are scored the same way in both units, so only the interior bins change). For the original
linking it varies by several nats: the exact 0s and 100s are scored by density at the nudged value in native
units, versus the bin's probability in discrete units, and the gap depends on how sharply the fitted shapes
pile up at the edges, which changes with θ. So the native ELBO differences **within** the original linking
also shift under the conversion.

In [8]:
pd.DataFrame({name: {'shift @ median': main.loc[name, 'shift @ median'], 'mean': sh.mean(), 'sd': sh.std(ddof=1),
                     'min': sh.min(), 'max': sh.max()} for name, sh in SHIFTS.items() if name in MODELS}).T.round(2)

,shift @ median,mean,sd,min,max
"ZOIB, β fitted",-24310.23,-24310.28,0.16,-24310.51,-24309.50
"original, β fixed (≈29)",-27420.17,-27417.35,9.62,-27430.34,-27394.31
"original, β fitted",-27425.54,-27420.13,7.01,-27428.47,-27398.23


## 4. Supplementary: every fit in one table

All full and null fits (including β = 3), ranked by the converted ELBO. The nulls are computed in section 2b.

In [9]:
cols = ['β', 'VBMC status', 'ELBO (native)', 'E_q[shift]', 'SE', 'ELBO (discrete, converted)', 'log Z @ median (discrete)']
out = both[cols].copy()
out['Δ ELBO vs best'] = out['ELBO (discrete, converted)'] - out['ELBO (discrete, converted)'].max()
out.sort_values('ELBO (discrete, converted)', ascending=False).round(1)

,β,VBMC status,ELBO (native),E_q[shift],SE,"ELBO (discrete, converted)",log Z @ median (discrete),Δ ELBO vs best
model,,,,,,,,
"ZOIB, β fitted",25.9,probable,-1724.9,-24310.3,0.0,-26035.2,-26030.1,0.0
"ZOIB, β fitted: null",22.8,probable,-1731.8,-24310.4,0.0,-26042.2,-26037.0,-7.0
"original, β fitted: null",0.7,probable,970.0,-27389.2,1.6,-26419.2,-26409.5,-384.0
"original, β fixed at 3: null",3.0,probable,966.1,-27396.8,1.9,-26430.7,-26428.8,-395.6
"original, β fixed (≈29): null",28.9,probable,963.3,-27395.6,1.7,-26432.3,-26431.5,-397.1
"original, β fixed at 3",3.0,no,976.5,-27409.1,1.9,-26432.6,-26424.0,-397.4
"original, β fitted",0.9,no,983.3,-27420.1,1.1,-26436.8,-26436.8,-401.6
"original, β fixed (≈29)",28.9,probable,974.5,-27417.3,1.5,-26442.8,-26438.6,-407.6


## 5. Caveats

- **The linking parameters are profiled (maximized), not integrated.** The ELBO integrates over θ only. The
  original linking has 4 parameters per feature (60 total) and ZOIB has 8 (120 total), so the ZOIB model gets 60
  more free parameters that the evidence doesn't charge for. A BIC-style charge of ½·log(N) per parameter would be
  about ½·60·log(6015) ≈ 261 nats against ZOIB; the cell below shows the table with that rough penalty. It's a
  sensitivity check, not a derived correction.
- **The original linking's shapes are profiled under the density likelihood**, then scored under the discrete one.
  Re-profiling them under the discrete likelihood could only raise the original model's discrete log Z, so its
  number here is a slight underestimate.
- **Convergence.** The "original, β fitted" fit is VBMC status "no" (hit the 400-eval cap), so its ELBO is less
  reliable than the others.
- **The ZOIB fit's weak Beta(1.5, 1.5) penalty on π** enters its linking fit only, not log Z (as in `inference.py`).

In [10]:
n_link = {'zoib': 8 * geom['J'], 'orig': 4 * geom['J']}
pen = pd.DataFrame({name: {'linking params (profiled)': n_link[link],
                           'ELBO (discrete, converted)': main.loc[name, 'ELBO (discrete, converted)'],
                           'minus ½·k·log N': main.loc[name, 'ELBO (discrete, converted)'] - 0.5 * n_link[link] * np.log(N)}
                    for name, (link, _) in MODELS.items()}).T
pen['Δ vs best (penalized)'] = pen['minus ½·k·log N'] - pen['minus ½·k·log N'].max()
pen.round(1)

,linking params (profiled),"ELBO (discrete, converted)",minus ½·k·log N,Δ vs best (penalized)
"ZOIB, β fitted",120.0,-26035.2,-26557.3,0.0
"original, β fixed (≈29)",60.0,-26442.8,-26703.9,-146.6
"original, β fitted",60.0,-26436.8,-26697.9,-140.5
